# Libraries and OpenAI API

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Change directory to this folder
%cd /content/drive/MyDrive/GenAI/Unstructured and Multimodal Data/Unstructured Data

/content/drive/MyDrive/GenAI/Unstructured and Multimodal Data/Unstructured Data


In [ ]:
# Import the userdata module from Google Colab
from google.colab import userdata
# Retrieve the API key stored under 'genai_course' from Colab's userdata
api_key = userdata.get('OPENAI_API_KEY3')

In [12]:
# Install libraries
!pip install langchain-community unstructured langchain-openai faiss-cpu -q
# Excel
!pip install msoffcrypto-tool -q
# Word
!pip install python-docx -q
!pip install --upgrade nltk -q
# PPT
!pip install python-pptx -q
# EPUB
!pip install pypandoc pandoc -q

# PDF
!pip install pymupdf pdfminer.six pillow_heif unstructured_inference unstructured_pytesseract pi_heif pdf2image -q
!apt-get install poppler-utils -q
!apt install tesseract-ocr -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
zsh:1: command not found: apt-get
zsh:1: command not found: apt


In [ ]:
# Set API key in the os
import os
os.environ["OPENAI_API_KEY"] = api_key

# Configurations

In [ ]:
# --- GLOBAL SETTINGS ---
# Embedding model
EMBEDDING_MODEL = "text-embedding-3-large"

# GPT model
GPT_MODEL = "gpt-5-nano"

In [ ]:
# import essential libraries
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.vectorstores.faiss import FAISS
from IPython.display import display, Markdown

In [14]:
import pandas as pd
from langchain_community.document_loaders import UnstructuredExcelLoader
from langchain_core.documents import Document

# Helpful functions

In [ ]:
# Build  function for excel data
def prepare_excel(file_path):
  # import the data
  df = pd.read_excel(file_path)

  # One Document per row
  docs = [
      Document(
          page_content = "\n".join(f"{col}: {val}" for col, val in row.items()),
          metadata = {"row_index": int(i)}
      )
      for i, row in df.iterrows()
  ]
  return docs

In [ ]:
# Prepare function for retrieval
def retrieval(docs, chunk_size = 800, chunk_overlap = 400, embedding_model ="text-embedding-3-large" ):
  # Splitting into chunks
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size = chunk_size,
      chunk_overlap = chunk_overlap)
  chunks = text_splitter.split_documents(docs)

  # Start with embeddings
  embeddings = OpenAIEmbeddings(model = EMBEDDING_MODEL)

  # Store in the Vector Store
  db_faiss = FAISS.from_documents(chunks, embeddings)
  return db_faiss

In [ ]:
# Function to generate answer
def generate_answer(db_faiss, query, k = 10, GPT_MODEL = "gpt-5-nano"):
  # Retrieve the context using faiss
  retrieved_docs = db_faiss.similarity_search_with_score(query, k = k)

  # Join the context
  context = "\n".join([doc.page_content for doc, score in retrieved_docs])

  # Build prompt
  prompt = f"""
  Answer the {query} based on this retrieved context: {context}.
  If you don't know, say you don't know.
  """

  # Call the OpenAI API
  model = ChatOpenAI(model = GPT_MODEL)
  answer = model.invoke(prompt)
  return display(Markdown(answer.content))

# Excel

In [ ]:
import pandas as pd
from langchain_community.document_loaders import UnstructuredExcelLoader
from langchain_core.documents import Document

In [ ]:
# import the data
df = pd.read_excel("Reviews.xlsx")

In [ ]:
# One Document per row
docs = [
    Document(
        page_content = "\n".join(f"{col}: {val}" for col, val in row.items()),
        metadata = {"row_index": int(i)}
    )
    for i, row in df.iterrows()
]
docs[:5]

[Document(metadata={'row_index': 0}, page_content='Course Name: Master Python for Data Analysis and Business Analytics 2024\nStudent Name: Gaurav Mehra\nTimestamp: 2024-08-21 06:46:55+00:00\nRating: 4.0\nComment: nan'),
 Document(metadata={'row_index': 1}, page_content='Course Name: Master Python for Data Analysis and Business Analytics 2024\nStudent Name: Harigovind S\nTimestamp: 2024-08-21 04:35:13+00:00\nRating: 5.0\nComment: nan'),
 Document(metadata={'row_index': 2}, page_content='Course Name: Data Literacy and Business Analytics for Business Leaders\nStudent Name: Celine Jayme\nTimestamp: 2024-08-21 01:42:37+00:00\nRating: 4.0\nComment: nan'),
 Document(metadata={'row_index': 3}, page_content='Course Name: Decision Making with Problem Solving & Critical Thinking\nStudent Name: Donovan Smith\nTimestamp: 2024-08-20 20:02:59+00:00\nRating: 4.0\nComment: nan'),
 Document(metadata={'row_index': 4}, page_content='Course Name: Econometrics and Statistics for Business in R & Python\nStud

In [ ]:
# Optional - slit long rows
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap = 400)
chunks = text_splitter.split_documents(docs)
chunks[:5]

[Document(metadata={'row_index': 0}, page_content='Course Name: Master Python for Data Analysis and Business Analytics 2024\nStudent Name: Gaurav Mehra\nTimestamp: 2024-08-21 06:46:55+00:00\nRating: 4.0\nComment: nan'),
 Document(metadata={'row_index': 1}, page_content='Course Name: Master Python for Data Analysis and Business Analytics 2024\nStudent Name: Harigovind S\nTimestamp: 2024-08-21 04:35:13+00:00\nRating: 5.0\nComment: nan'),
 Document(metadata={'row_index': 2}, page_content='Course Name: Data Literacy and Business Analytics for Business Leaders\nStudent Name: Celine Jayme\nTimestamp: 2024-08-21 01:42:37+00:00\nRating: 4.0\nComment: nan'),
 Document(metadata={'row_index': 3}, page_content='Course Name: Decision Making with Problem Solving & Critical Thinking\nStudent Name: Donovan Smith\nTimestamp: 2024-08-20 20:02:59+00:00\nRating: 4.0\nComment: nan'),
 Document(metadata={'row_index': 4}, page_content='Course Name: Econometrics and Statistics for Business in R & Python\nStud

In [ ]:
# Start with embeddings
embeddings = OpenAIEmbeddings(model = EMBEDDING_MODEL)

In [ ]:
# Store in the Vector Store
db_faiss = FAISS.from_documents(chunks, embeddings)
db_faiss

In [ ]:
# Test the Retrieval System
query = "Give me the best reviews with comments"

# Retrieve the context using faiss
retrieved_docs = db_faiss.similarity_search_with_score(query, k = 3)
retrieved_docs

[(Document(id='14131d2b-40cc-4151-b95a-05a86391b850', metadata={'row_index': 3857}, page_content="Course Name: Econometrics and Statistics for Business in R & Python\nStudent Name: Moaz alhomsi\nTimestamp: 2020-06-27 05:09:22+00:00\nRating: 5.0\nComment: This course is simply the best econometrics course out there. Diogo knows what he is doing, real life case studies, no nonsense theories and complications. I highly recommend this course.\nBut including some math could've been great, maybe adding some formulas as papers in the next update will be GREAT. Thank you Diogo for the amazing course, waiting for more !"),
  np.float32(1.2891192)),
 (Document(id='b88c1c73-e3a4-45ca-9a2d-d29db458704f', metadata={'row_index': 1465}, page_content='Course Name: Master Time Series Analysis and Forecasting with Python 2024\nStudent Name: Prabhat k mishra\nTimestamp: 2023-08-08 17:41:22+00:00\nRating: 5.0\nComment: Very Nice. Want some more and more  Business related Topics.'),
  np.float32(1.3175731)

In [ ]:
# Join the context
context = "\n".join([doc.page_content for doc, score in retrieved_docs])
display(Markdown(context))

Course Name: Econometrics and Statistics for Business in R & Python
Student Name: Moaz alhomsi
Timestamp: 2020-06-27 05:09:22+00:00
Rating: 5.0
Comment: This course is simply the best econometrics course out there. Diogo knows what he is doing, real life case studies, no nonsense theories and complications. I highly recommend this course.
But including some math could've been great, maybe adding some formulas as papers in the next update will be GREAT. Thank you Diogo for the amazing course, waiting for more !
Course Name: Master Time Series Analysis and Forecasting with Python 2024
Student Name: Prabhat k mishra
Timestamp: 2023-08-08 17:41:22+00:00
Rating: 5.0
Comment: Very Nice. Want some more and more  Business related Topics.
Course Name: Statistics for Business Analytics: Data Analysis with Excel
Student Name: Pankaj sharma
Timestamp: 2024-02-23 06:30:41+00:00
Rating: 5.0
Comment: nice one of the best for freshers

In [ ]:
# Build prompt
prompt = f"""
Answer the {query} based on this retrieved context: {context}.
If you don't know, say you don't know.
"""

In [ ]:
# Call the OpenAI API
model = ChatOpenAI(model = GPT_MODEL)
answer = model.invoke(prompt)
display(Markdown(answer.content))

Here are the best reviews from the provided context, with comments included:

- Moaz alhomsi — Course: Econometrics and Statistics for Business in R & Python — Timestamp: 2020-06-27 05:09:22+00:00 — Rating: 5.0
  Comment: "This course is simply the best econometrics course out there. Diogo knows what he is doing, real life case studies, no nonsense theories and complications. I highly recommend this course. But including some math could've been great, maybe adding some formulas as papers in the next update will be GREAT. Thank you Diogo for the amazing course, waiting for more !"

- Prabhat k mishra — Course: Master Time Series Analysis and Forecasting with Python 2024 — Timestamp: 2023-08-08 17:41:22+00:00 — Rating: 5.0
  Comment: "Very Nice. Want some more and more  Business related Topics."

- Pankaj sharma — Course: Statistics for Business Analytics: Data Analysis with Excel — Timestamp: 2024-02-23 06:30:41+00:00 — Rating: 5.0
  Comment: "nice one of the best for freshers. If you don't know, say you don't know."

If you’d like, I can summarize themes across these reviews (e.g., emphasis on practical, real-life case studies; desire for more math content; accessibility for beginners) or format them differently.

In [ ]:
# Test the functions
docs = prepare_excel("Reviews.xlsx")
db_faiss = retrieval(docs)
generate_answer(db_faiss, "Give me the best reviews with comments")

Here are the top-rated reviews (all 5.0) with quotes pulled from the provided context:

- Moaz alhomsi — Econometrics and Statistics for Business in R & Python
  "This course is simply the best econometrics course out there. Diogo knows what he is doing, real life case studies, no nonsense theories and complications. I highly recommend this course. But including some math could've been great, maybe adding some formulas as papers in the next update will be GREAT. Thank you Diogo for the amazing course, waiting for more !"

- Antonio Rodriguez Andres — Econometrics and Statistics for Business in R & Python
  "Wonderful course, excellent instructor, easily reachable, always ready to help students, he knows what he is doing, and with useful applications for the real life. Without doubt, one of the best instructors in Udemy"

- Abhimanyu Trakroo — Master Time Series Analysis and Forecasting with Python 2024
  "I have taken couple of courses by this instructor - Econometrics, and Time series (this course). It is amazing and refreshing to see how he manages to convey the essence of the topic and its practical implementation in programming, without making the lectures too long. And I have liked that he is always quick to resolve queries and doubts on lectures. Continue the good work!! You make learning interesting."

- Deepak Giri — Master Time Series Analysis and Forecasting with Python 2024
  "This is one of the best courses on learning various forecasting models in python. Diogo (instructor) is extremely clear in the course content, the templates provided as spot on and most importantly, Diogo is always quick to answer any questions coming from the trainees. Thanks so much for this knowledge, kudos to Diogo and the team!."

- Quinn McCann — Master Time Series Analysis and Forecasting with Python 2024
  "Excellent Course-the best of a half dozen time-series/forecasting classes I've taken on Udemy. Facebook Prophet, and the combination with XGBoost especially stand out. I hope Diogo does another course."

- Prabhat k mishra — Master Time Series Analysis and Forecasting with Python 2024
  "Very Nice. Want some more and more  Business related Topics."

- Pankaj Sharma — Statistics for Business Analytics: Data Analysis with Excel
  "nice one of the best for freshers"

- Yatharth Negi — Business Data Analytics & Intelligence with Python
  "It nice for me"

- Ege Oyan — Master Time Series Analysis and Forecasting with Python 2024
  "Getting better and better"

If you’d like, I can filter these further (e.g., only from a specific course) or format them as a short testimonial block for a webpage.

# Word documents

In [ ]:
# import modules
import nltk
from langchain_community.document_loaders import UnstructuredWordDocumentLoader
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
# Load the word document
def load_word_document(file_path):
    loader = UnstructuredWordDocumentLoader(file_path)
    docs = loader.load()
    return docs
docs_word = load_word_document("Declaration of independence.docx")
docs_word

[Document(metadata={'source': 'Declaration of independence.docx'}, page_content="In Congress, July 4, 1776\n\nThe unanimous Declaration of the thirteen united States of America,\xa0When in the Course of human events, it becomes necessary for one people to dissolve the political bands which have connected them with another, and to assume among the powers of the earth, the separate and equal station to which the Laws of Nature and of Nature's God entitle them, a decent respect to the opinions of mankind requires that they should declare the causes which impel them to the separation.\n\nWe hold these truths to be self-evident, that all men are created equal, that they are endowed by their Creator with certain unalienable Rights, that among these are Life, Liberty and the pursuit of Happiness.--That to secure these rights, Governments are instituted among Men, deriving their just powers from the consent of the governed, --That whenever any Form of Government becomes destructive of these en

In [ ]:
# Apply the retrieval function
db_faiss = retrieval(docs_word)

In [ ]:
# apply the generative function
generate_answer(db_faiss, "What is the purpose of this declaration?")

The purpose is to formally declare the American colonies as free and independent states, dissolving all political ties with the British Crown. It asserts the right to self-government and to engage in war, peace, alliances, and commerce, and it justifies this break by listing grievances against the Crown and articulating a philosophy of government based on the consent of the governed and inherent rights.

In [ ]:
# Test with a silly question
generate_answer(db_faiss, "I like chocolate")

I don’t know. The retrieved context doesn’t mention chocolate or anyone’s preference for it.

# PPT

In [ ]:
# essential function
from langchain_community.document_loaders import UnstructuredPowerPointLoader

In [ ]:
def prepare_ppt(file_path):
  loader = UnstructuredPowerPointLoader(file_path)
  docs = loader.load()
  return docs
docs_ppt = prepare_ppt("Bitte pitch deck EN.pptx")

In [ ]:
# Prepare the database
db_faiss = retrieval(docs_ppt)

In [ ]:
# Test the RAG system
generate_answer(db_faiss, "What is the purpose of this presentation?")


- To introduce Bitte’s digital menu solution and show how it solves common restaurant menu problems (unappealing design, lack of interactivity, dependence on third parties, high costs).
- To communicate Bitte’s vision and mission: provide innovative solutions that enhance customer experience and improve operational/financial efficiency.
- To highlight the product’s value: appealing, brand-aligned layouts, mobile access, interactive recommendations, AI-driven insights, and easy sharing.
- To present market opportunity and traction: large TAM/SAM/SOM, MVP in use by 12 restaurants with more in production, initial funding, and a clear go-to-market plan.
- To outline the business model and next steps: revenue streams (freemium, commissions, expanded services), roadmap, and funding goals.

In [ ]:
# Test with a silly query
generate_answer(db_faiss, "I like chocolate")

I don’t know. The retrieved context is about digital restaurant menus, AI-driven management, and business models, and it doesn’t mention chocolate or anyone’s taste. If you’d like, I can help brainstorm how a digital menu could promote chocolate desserts or tailor offers around them.

# EPUB

In [ ]:
# import essential modules and classes
from langchain_community.document_loaders import UnstructuredEPubLoader
import pypandoc
pypandoc.download_pandoc()

In [ ]:
# Build the function to prepare the pub
def prepare_epub(file_path):
  loader = UnstructuredEPubLoader(file_path)
  docs = loader.load()
  return docs
docs_epub = prepare_epub("Alice’s Adventures in Wonderland.epub")

In [ ]:
# PRepare the FAISS DB
db_faiss = retrieval(docs_epub)

In [ ]:
# Test the RAG
generate_answer(db_faiss, "Tell me the key points of this book")
#

Here are the key points you can extract from the retrieved context (primarily CH I and CH VII) of Alice's Adventures in Wonderland:

- Protagonist and mood
  - Alice is a curious, imaginative girl who is bored and tired of adult world and lacking engaging pictures or conversations in the book she sees.

- Inciting event and gateway to Wonderland
  - She notices a White Rabbit with pink eyes, follows it, and falls down a rabbit-hole, entering a strange, dreamlike world.

- Motivation and early obstacles
  - In CH I, she searches for a way to escape boredom and to fit through doors, discovering a bottle labeled “DRINK ME” that suggests she can alter her size, a key on a table, and a door she can’t reach.

- Meta-fictional setup
  - The narrative hints at playful contradictions between stories and rules: Alice questions the usefulness of books without pictures or conversations, signaling the book’s own love of wordplay, rules, and imaginative encounters.

- Early encounters in Wonderland (themes and style)
  - In this early material, Wonderland is a place where logic is inverted and rules are arbitrary, showcased by conversations with talking animals and odd behavior (e.g., the Mock Turtle and Gryphon debating adventure and explanations).
  - The Mock Turtle’s dry, pedantic recital (“Explain all that”) and the Gryphon’s corrective push for “adventures first” illustrate the book’s humorous subversion of normal storytelling expectations.

- The Mad Tea-Party (CH VII) specifics
  - A chaotic scene with the March Hare, the Hatter, and a sleeping Dormouse; they crowd around a small table and complain about space.
  - The Hatter and Hare engage in nonsensical, circular dialogue; the tea party is a parody of social etiquette and formal conversation.
  - Alice feels out of place and challenges the strange social dynamics, highlighting the clash between adult pretensions and child perspective.

- Tone and form
  - The book uses episodic vignettes, satire, and wordplay to create a whimsical, illogical, and satirical world that mocks adult rules, language, and seriousness.

- What this excerpt suggests about the book’s arc
  - Expect a series of surreal episodes (riddles, size changes, talking creatures, and absurd encounters) rather than a linear plot.
  - Major themes to anticipate include curiosity versus authority, language and meaning, identity, and the exploration of a child’s perspective through dreamlike adventures.

If you’d like, I can expand this with additional chapters or pull out more themes and episodes based on other parts of the text.

# PDFs

In [13]:
# import libraries
from langchain_community.document_loaders import UnstructuredPDFLoader
from pdfminer import psparser

/Users/akshaytelang/Documents/GenAI/Unstructured and Multimodal Data/Unstructured Data/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Build function to prepare PDF
def prepare_pdf(file_path):
  loader = UnstructuredPDFLoader(file_path)
  docs = loader.load()
  return docs
docs_pdf = prepare_pdf("Famous old receipts.pdf")

In [ ]:
# Build the FAISS DB
db_faiss = retrieval(docs_pdf)

In [ ]:
# test the RAG
generate_answer(db_faiss, "Find me the tp 3 recipes that would impress my guests")

Top 3 from the context that would most impress guests:

- Caviar on Toast
  - Source: Contributed by Mrs. Charles A. Farnum, Philadelphia, Pa.
  - Why it impresses: Luxurious caviar served over toast makes an elegant, instantly impressive starter.
  - Recipe detail (as given): Prepare rounds of white bread toast. Place over the fire for a minute or two, then layer with two large tablespoons of caviar and one tablespoon of cream, stirring the mixture while heating. Pour this mixture over the toast.

- Lobster Cutlets
  - Source: Mrs. M. Kim Miller.
  - Why it impresses: Lobster is a classic, sophisticated centerpiece; “cutlets” suggest a refined preparation that can be plated chicly.

- Sabot a la Creme au Gratin
  - Source: Mrs. Bradley T. Johnson.
  - Why it impresses: French-sounding, creamy gratin dish that signals a touch of haute cuisine and guests often associate with formal entertaining.

If you’d like, I can pull more details or adapt these for modern kitchens (e.g., simpler substitutions if fresh caviar isn’t available, or scaled-down portions for fewer guests).

https://python.langchain.com/docs/integrations/providers/unstructured/

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv('food_reviews_1k.csv')

np.int64(1)

In [ ]:
# --- Run the full pipeline ---
docs = prepare_excel_azure("Reviews.xlsx")
db_faiss = retrieval_azure(docs)
generate_answer_azure(db_faiss, "Give me the best reviews with comments")

In [ ]:
# Step 1: Load Excel — one Document per row
def prepare_excel_azure(file_path):
    df = pd.read_excel(file_path)
    docs = [
        Document(
            page_content="\n".join(f"{col}: {val}" for col, val in row.items()),
            metadata={"row_index": int(i)}
        )
        for i, row in df.iterrows()
    ]
    print(f"Loaded {len(docs)} rows as documents")
    return docs


# Step 2: Chunk + Embed + Store in FAISS
def retrieval_azure(docs, chunk_size=800, chunk_overlap=400):
    # Split long documents
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = text_splitter.split_documents(docs)
    print(f"Created {len(chunks)} chunks")

    # Azure embeddings — only difference from original is AzureOpenAIEmbeddings + deployment config
    embeddings = AzureOpenAIEmbeddings(
        azure_deployment=AZURE_EMBEDDING_DEPLOYMENT,
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        api_version=AZURE_API_VERSION
    )

    db_faiss = FAISS.from_documents(chunks, embeddings)
    print("FAISS vector store ready!")
    return db_faiss


# Step 3: Retrieve + Generate answer
def generate_answer_azure(db_faiss, query, k=10):
    # Retrieve top-k similar chunks
    retrieved_docs = db_faiss.similarity_search_with_score(query, k=k)
    context = "\n".join([doc.page_content for doc, score in retrieved_docs])

    prompt = f"""
    Answer the question: {query}
    Based on this retrieved context: {context}
    If the answer is not in the context, say you don't know.
    """

    # Azure chat model — only difference is AzureChatOpenAI + deployment config
    model = AzureChatOpenAI(
        azure_deployment=AZURE_CHAT_DEPLOYMENT,
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        api_version=AZURE_API_VERSION
    )
    answer = model.invoke(prompt)
    return display(Markdown(answer.content))

In [ ]:
import pandas as pd
from langchain_openai import AzureOpenAIEmbeddings, AzureChatOpenAI  # Azure variants
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from IPython.display import display, Markdown

In [ ]:
# Install required libraries (run once)
# langchain-openai already supports Azure — no extra package needed
# !pip install langchain-community langchain-openai faiss-cpu openpyxl -q

In [ ]:
import os

# --- Paste your Azure credentials here ---
os.environ["AZURE_OPENAI_API_KEY"]  = "your-azure-openai-key-here"
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://your-resource-name.openai.azure.com/"

# --- Deployment names from Azure OpenAI Studio → Deployments ---
AZURE_EMBEDDING_DEPLOYMENT = "text-embedding-3-large"  # your embedding deployment name
AZURE_CHAT_DEPLOYMENT      = "gpt-4o"                  # your chat deployment name

# --- API version (keep this, it works for all modern deployments) ---
AZURE_API_VERSION = "2024-08-01-preview"

# Excel with Azure OpenAI

Using Azure OpenAI instead of the standard OpenAI API.  
You need 3 things from your Azure Portal:
- **API Key** → Azure OpenAI resource → Keys and Endpoint
- **Endpoint** → same page (looks like `https://your-resource.openai.azure.com/`)
- **Deployment names** → Azure OpenAI Studio → Deployments (one for embeddings, one for chat)

In [ ]:
# Semantic Chunking
# SemanticChunker splits text at semantically meaningful boundaries using embeddings.
# Sentences that are topically similar are grouped into the same chunk.

!pip install langchain-experimental -q

from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

# --- Sample paragraph covering 3 distinct topics ---
sample_text = """
The solar system consists of the Sun and the objects that orbit it, including eight planets, 
their moons, and various smaller bodies such as asteroids and comets. Jupiter is the largest 
planet, while Mercury is the smallest. Earth is the only known planet to support life.

Machine learning is a branch of artificial intelligence that enables computers to learn from 
data without being explicitly programmed. Supervised learning uses labeled data to train models, 
while unsupervised learning finds hidden patterns in unlabeled data. Deep learning uses neural 
networks with many layers to solve complex problems.

The French Revolution began in 1789 and fundamentally transformed France's political landscape. 
It led to the rise of Napoleon Bonaparte and spread democratic ideals across Europe. The storming 
of the Bastille on July 14, 1789 became a powerful symbol of the uprising against monarchy.
"""

# --- Initialize SemanticChunker ---
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

semantic_splitter = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=70,
)

# --- Split and print chunks ---
chunks = semantic_splitter.create_documents([sample_text])

print(f"Total chunks: {len(chunks)}\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk.page_content.strip())
    print()